In [66]:
from langgraph.graph import StateGraph,START,END
from langchain_ollama import ChatOllama
from typing import TypedDict


In [67]:
class BatsmanState(TypedDict):
    runs:int
    balls:int
    fours:int
    sixes:int

    sr:float
    bpb:float
    boundary_percent:float
    summary:str

In [68]:
def calcualte_sr(state:BatsmanState) :
    sr=(state['runs']/state['balls'])*100
    return {'sr': sr}

In [69]:
def calcualte_bpb(state:BatsmanState):
    bpb=state['balls']/(state['fours']+state['sixes'])
    return {'bpb': bpb}

In [70]:
def calcualte_boundarypercentage(state):
    boundary_percent  = (
        ((state['fours']*4)+(state['sixes']*6))
        / state['runs']
    ) * 100

    return {
        "boundary_percent": boundary_percent
    }

In [71]:

def summary(state: BatsmanState):

    summary = f"""
Strike Rate - {state['sr']} \n
Balls per boundary - {state['bpb']} \n
Boundary percent - {state['boundary_percent']}
"""
    
    return {'summary': summary}

In [72]:
graph=StateGraph(BatsmanState)
graph.add_node('calcualte_sr',calcualte_sr)
graph.add_node('calcualte_bpb',calcualte_bpb)
graph.add_node('calcualte_boundarypercentage',calcualte_boundarypercentage)
graph.add_node('summary',summary)


graph.add_edge(START,'calcualte_sr')
graph.add_edge(START,'calcualte_bpb')
graph.add_edge(START,'calcualte_boundarypercentage')
graph.add_edge('calcualte_sr','summary')
graph.add_edge('calcualte_bpb','summary')
graph.add_edge('calcualte_boundarypercentage','summary')

graph.add_edge('summary',END)
work_flow=graph.compile()


In [73]:
inital_state={
    'runs':100,
    'balls':50,
    'fours':6,
    'sixes':4
}
work_flow.invoke(inital_state)

{'runs': 100,
 'balls': 50,
 'fours': 6,
 'sixes': 4,
 'sr': 200.0,
 'bpb': 5.0,
 'boundary_percent': 48.0,
 'summary': '\nStrike Rate - 200.0 \n\nBalls per boundary - 5.0 \n\nBoundary percent - 48.0\n'}